In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import warnings
warnings.filterwarnings("ignore")





In [3]:
# LOAD CSV FILES
train_df = pd.read_csv("/content/Sample_arvyax_reflective_dataset.xlsx - Dataset_120.csv")
test_df  = pd.read_csv("/content/arvyax_test_inputs_120.xlsx - Sheet1.csv")

In [4]:
# LOAD CSV FILES
for col in ["journal_text"]:
    train_df[col].fillna("", inplace=True)
    test_df[col].fillna("", inplace=True)
for col in ["sleep_hours", "energy_level", "stress_level"]:
    median = train_df[col].median()
    train_df[col].fillna(median, inplace=True)
    test_df[col].fillna(median, inplace=True)


In [5]:
# TEXT FEATURES
tfidf = TfidfVectorizer(max_features=5000)

X_train_text = tfidf.fit_transform(train_df["journal_text"])
X_test_text  = tfidf.transform(test_df["journal_text"])

In [6]:
# META FEATURES
meta_cols = ["sleep_hours", "energy_level", "stress_level"]

scaler = StandardScaler()

X_train_meta = scaler.fit_transform(train_df[meta_cols])
X_test_meta  = scaler.transform(test_df[meta_cols])

In [7]:
# COMBINE FEATURES
X_train = hstack([X_train_text, X_train_meta])
X_test  = hstack([X_test_text, X_test_meta])

In [8]:
# TARGETS (ONLY TRAIN)
y_state = train_df["emotional_state"]
y_int   = train_df["intensity"]

In [9]:
# MODEL TRAINING
state_model = LogisticRegression(max_iter=1000)
state_model.fit(X_train, y_state)

int_model = LogisticRegression(max_iter=1000)
int_model.fit(X_train, y_int)

LogisticRegression(max_iter=1000)

In [10]:
# PREDICTIONS (ON TEST)
state_preds = state_model.predict(X_test)
int_preds   = int_model.predict(X_test)

In [11]:
# CONFIDENCE & UNCERTAINTY
state_probs = state_model.predict_proba(X_test)
confidence = np.max(state_probs, axis=1)

uncertain_flag = (confidence < 0.5).astype(int)

In [12]:
# DECISION ENGINE
def decide_action(row, state, intensity):
    stress = row["stress_level"]
    energy = row["energy_level"]
    time = row["time_of_day"]

    if stress > 7 and energy < 4:
        return "rest", "now"

    elif stress > 7:
        return "box_breathing", "now"

    elif energy > 7 and intensity <= 2:
        return "deep_work", "within_15_min"

    elif state in ["sad", "anxious"]:
        return "journaling", "later_today"

    elif time == "night":
        return "sleep", "tonight"

    else:
        return "light_planning", "later_today"

In [13]:
# BUILD OUTPUT
results = []

for i in range(len(test_df)):
    row = test_df.iloc[i]

    what, when = decide_action(row, state_preds[i], int_preds[i])

    results.append([
        row["id"],
        state_preds[i],
        int_preds[i],
        confidence[i],
        uncertain_flag[i],
        what,
        when
    ])

In [14]:
# SAVE FINAL CSV
output = pd.DataFrame(results, columns=[
    "id",
    "predicted_state",
    "predicted_intensity",
    "confidence",
    "uncertain_flag",
    "what_to_do",
    "when_to_do"
])

output.to_csv("predictions.csv", index=False)
